
# Cálculo de asistencia de diputados (Chile)

Este notebook calcula el **porcentaje de asistencia** de cada diputado a partir
de dos matrices de votación (`matriz__periodo_2022_26` y `matriz__periodo_2014_18`)
y lo empareja con la base en formato *long* de ideología
(`ideologia_congreso_chile_2002_2026_long_format`) usando `Period` y `Name`.

**Fórmula de asistencia**

\[
\text{Asistencia} = \frac{\text{Afirmativo} + \text{En Contra}}{\text{cualquier otra entrada de voto}}
\]

Es decir: en el denominador se cuentan **todas** las celdas de votación de un
diputado (todas las columnas de votos, sin importar el valor: Afirmativo, En
Contra, Abstención, Dispensado, NA, vacío, etc. — todo lo que no sean las
columnas identificadoras `n`, `DiputadoId`, `Nombre`, `ApellidoPaterno`,
`ApellidoMaterno`).

**Mapeo de períodos**

| Archivo matriz                  | Period en la base long |
|----------------------------------|-------------------------|
| `matriz__periodo_2022_26`        | `Periodo_6`             |
| `matriz__periodo_2014_18`        | `Periodo_4`             |

**Clave de emparejamiento:** `Period` + `Name`, donde `Name` se construye
concatenando `Nombre` + `ApellidoPaterno` + `ApellidoMaterno` (columnas C, D, E),
normalizado a formato "Primera Letra Mayúscula" separado por un solo espacio,
para que calce con la columna `Name` de la base long.


In [1]:

# 1. Librerías
import re
import unicodedata
from pathlib import Path

import pandas as pd



## 1. Rutas de los archivos

Ajusta estas rutas a donde tengas guardados tus archivos. Se aceptan `.csv` o
`.xlsx` — el loader detecta la extensión automáticamente.


In [4]:

# Carpeta donde están tus archivos (ajusta si es necesario)
DATA_DIR = Path("C:\\Users\\merri\\OneDrive - Universidad de Chile\\'26\\Seminario de Investigacion\\Evidence and estimations")

PATH_2022_26 = DATA_DIR / "matriz__periodo_2022_26.csv"   # -> Periodo_6
PATH_2014_18 = DATA_DIR / "matriz__periodo_2014_18.csv"   # -> Periodo_4
PATH_LONG    = DATA_DIR / "ideologia_congreso_chile_2002_2026_long_format.csv"

# Columnas identificadoras (no son votos) en las matrices de asistencia
ID_COLS = ["n", "DiputadoId", "Nombre", "ApellidoPaterno", "ApellidoMaterno"]



## 2. Funciones auxiliares

### 2.1 Corrección de encoding (mojibake)

Algunas bases traen nombres con caracteres corruptos como `MarÃ­a` (debería ser
`María`) o `RenÃ©` (debería ser `René`). Esto ocurre típicamente cuando un
archivo guardado en **UTF-8** fue reinterpretado como **Latin-1 / cp1252** (o
viceversa) en algún paso previo (Excel, copiar/pegar, etc.).

La función `fix_mojibake` detecta ese patrón específico y lo revierte:
intenta "re-codificar" el string como si hubiera sido mal decodificado y, si el
resultado es válido y tiene más sentido (sin caracteres de reemplazo `�`), lo
usa. Si el texto ya está bien, lo deja intacto.


In [5]:

def fix_mojibake(text):
    '''Repara nombres con doble-encoding tipo MarA-a -> Maria, RenA(c) -> Rene.

    Estrategia: si el texto contiene el patron tipico de UTF-8 mal
    decodificado como Latin-1 (secuencias 'A' + vocal/consonante especial,
    o 'A'), se intenta:  text.encode('latin-1').decode('utf-8')
    Si funciona sin errores y no introduce caracteres de reemplazo, se usa
    el resultado corregido. En caso contrario se devuelve el texto original.
    '''
    if not isinstance(text, str) or text == "":
        return text

    # Señales típicas de mojibake UTF-8 -> Latin-1
    mojibake_markers = ("Ã", "Â", "â€", "�")
    if not any(marker in text for marker in mojibake_markers):
        return text

    try:
        repaired = text.encode("latin-1").decode("utf-8")
        if "�" not in repaired:
            return repaired
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass

    # Intento alternativo: cp1252 -> utf-8 (variante común en exportes de Excel)
    try:
        repaired = text.encode("cp1252").decode("utf-8")
        if "�" not in repaired:
            return repaired
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass

    return text



### 2.2 Normalización de nombres (formato `Name`)

Convierte cualquier variante de capitalización/espaciado a
**"Primera Letra Mayúscula"**, separado por un solo espacio, preservando tildes
y la letra `ñ`. Ejemplos:

- `"MARÍA JOSÉ"` -> `"María José"`
- `"  josé   miguel  "` -> `"José Miguel"`
- `"o'higgins"` -> `"O'Higgins"`


In [6]:

def normalize_name_part(text):
    '''Limpia espacios y capitaliza cada palabra de un fragmento de nombre.'''
    if not isinstance(text, str):
        return ""
    text = fix_mojibake(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)  # colapsa espacios múltiples/tabs
    if text == "":
        return ""

    def cap_word(word):
        # Maneja apóstrofes y guiones dentro de la palabra (O'Higgins, Pérez-Díaz)
        parts = re.split(r"([-'])", word)
        return "".join(p.capitalize() if p not in ("-", "'") else p for p in parts)

    return " ".join(cap_word(w) for w in text.split(" "))


def build_full_name(nombre, apellido_paterno, apellido_materno):
    '''Concatena Nombre + ApellidoPaterno + ApellidoMaterno en formato Name.'''
    partes = [
        normalize_name_part(nombre),
        normalize_name_part(apellido_paterno),
        normalize_name_part(apellido_materno),
    ]
    partes = [p for p in partes if p]  # descarta vacíos
    return " ".join(partes)


def normalize_for_matching(text):
    '''Version sin tildes/mayusculas, solo para hacer el merge mas robusto
    ante pequenas diferencias de tildes entre bases (no se usa como columna
    final Name, solo como llave auxiliar de emparejamiento).'''
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    return text.lower().strip()



### 2.3 Carga robusta de archivos (CSV/XLSX con detección de encoding)


In [7]:

def load_table(path):
    '''Carga un csv o xlsx probando varios encodings comunes si es CSV.'''
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        return pd.read_excel(path, dtype=str)

    encodings_to_try = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
    last_err = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(path, dtype=str, encoding=enc, keep_default_na=False)
            return df
        except (UnicodeDecodeError, LookupError) as e:
            last_err = e
            continue
    raise last_err



## 3. Cálculo de asistencia por matriz

`compute_attendance` recibe un DataFrame de matriz (una fila por diputado,
columnas C/D/E = Nombre/ApellidoPaterno/ApellidoMaterno, y el resto de
columnas = votaciones), y devuelve un DataFrame con:

- `Name`: nombre normalizado (Primera Mayúscula, separado por un espacio)
- `Period`: `Periodo_6` o `Periodo_4` según el archivo
- `Afirmativo`, `En_Contra`, `Total_Votaciones`: conteos
- `Asistencia`: `(Afirmativo + En Contra) / Total_Votaciones`


In [8]:

def compute_attendance(df_raw, period_label, id_cols=ID_COLS):
    '''Calcula la asistencia por diputado para una matriz de votacion.

    Asistencia = (Afirmativo + En Contra) / (cualquier otra entrada de voto)
    El denominador cuenta TODAS las columnas de votacion (todas las columnas
    que no son identificadoras), sin importar su valor.
    '''
    df = df_raw.copy()

    # Arregla encoding en las columnas de nombre antes de todo
    for col in ("Nombre", "ApellidoPaterno", "ApellidoMaterno"):
        if col in df.columns:
            df[col] = df[col].map(fix_mojibake)

    # Columna Name (C + D + E) en formato Primera Letra Mayúscula
    df["Name"] = df.apply(
        lambda r: build_full_name(
            r.get("Nombre", ""), r.get("ApellidoPaterno", ""), r.get("ApellidoMaterno", "")
        ),
        axis=1,
    )

    # Columnas de votación = todas menos las identificadoras
    vote_cols = [c for c in df.columns if c not in id_cols and c != "Name"]

    votes = df[vote_cols]

    afirmativo = (votes == "Afirmativo").sum(axis=1)
    en_contra = (votes == "En Contra").sum(axis=1)
    total_votaciones = votes.shape[1]  # cualquier otra entrada = todas las columnas de voto

    out = pd.DataFrame({
        "Name": df["Name"],
        "Period": period_label,
        "Afirmativo": afirmativo,
        "En_Contra": en_contra,
        "Total_Votaciones": total_votaciones,
    })
    out["Asistencia"] = (out["Afirmativo"] + out["En_Contra"]) / out["Total_Votaciones"]

    # Llave auxiliar sin tildes/mayúsculas para un merge más tolerante
    out["Name_key"] = out["Name"].map(normalize_for_matching)

    return out



## 4. Ejecutar el cálculo para ambas matrices


In [9]:

df_2022_26_raw = load_table(PATH_2022_26)
df_2014_18_raw = load_table(PATH_2014_18)

asistencia_2022_26 = compute_attendance(df_2022_26_raw, period_label="Periodo_6")
asistencia_2014_18 = compute_attendance(df_2014_18_raw, period_label="Periodo_4")

asistencia_total = pd.concat([asistencia_2022_26, asistencia_2014_18], ignore_index=True)
asistencia_total.head()


,Name,Period,Afirmativo,En_Contra,Total_Votaciones,Asistencia,Name_key
0,María Candelaria Acevedo Sáez,Periodo_6,2393,776,3393,0.933982,maria candelaria acevedo saez
1,Eric Aedo Jeldres,Periodo_6,2157,461,3393,0.771589,eric aedo jeldres
2,Yovana Ahumada Palma,Periodo_6,1890,699,3393,0.763042,yovana ahumada palma
3,Jorge Alessandri Vergara,Periodo_6,1284,769,3393,0.605069,jorge alessandri vergara
4,René Alinco Bustos,Periodo_6,1860,404,3393,0.667256,rene alinco bustos



## 5. Cargar la base long de ideología y emparejar por `Period` + `Name`

Se hace primero un intento de merge exacto por `Period` + `Name`. Para los
que no calcen exactamente (por ejemplo, diferencias de tildes u orden de
apellidos), se intenta un segundo emparejamiento usando la llave normalizada
sin tildes (`Name_key`), y se reporta cuántos quedaron sin emparejar.


In [10]:

df_long_raw = load_table(PATH_LONG)

# Arregla posible mojibake también en la base long, por si acaso
if "Name" in df_long_raw.columns:
    df_long_raw["Name"] = df_long_raw["Name"].map(fix_mojibake)

df_long = df_long_raw.copy()
df_long["Name_key"] = df_long["Name"].map(normalize_for_matching)

df_long.head()


,n,Id,Name,Period,dim_IC_low,dim_ideology,dim_IC_high,wnom_ideology,bay_ic_low,bay_ideology,bay_ic_high,quant_parties,party_1,party_2,party_3,party_4,Sex,Birth_date,Name_key
0,1,74,Jaime Naranjo Ortiz,Periodo_1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,Masculino,1951-01-12T00:00:00,jaime naranjo ortiz
1,2,74,Jaime Naranjo Ortiz,Periodo_2,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,Masculino,1951-01-12T00:00:00,jaime naranjo ortiz
2,3,74,Jaime Naranjo Ortiz,Periodo_3,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,Masculino,1951-01-12T00:00:00,jaime naranjo ortiz
3,4,74,Jaime Naranjo Ortiz,Periodo_4,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,Masculino,1951-01-12T00:00:00,jaime naranjo ortiz
4,5,74,Jaime Naranjo Ortiz,Periodo_5,-0.668336246703293,-0.638761040942748,-0.609185835182204,-0.659219980239868,-0.676354548635916,-0.720716509444946,-0.781002000825575,1,PS,NA,NA,NA,Masculino,1951-01-12T00:00:00,jaime naranjo ortiz


In [ ]:

# --- Merge 1: exacto por Period + Name ---
merged = df_long.merge(
    asistencia_total.drop(columns=["Name_key"]),
    on=["Period", "Name"],
    how="left",
    suffixes=("", "_asistencia"),
)

sin_match = merged["Asistencia"].isna().sum()
print(f"Filas sin emparejar tras el merge exacto (Period + Name): {sin_match} de {len(merged)}")


In [ ]:

# --- Merge 2 (fallback): para los que no calzaron, reintentar con Name_key ---
if sin_match > 0:
    faltantes_idx = merged[merged["Asistencia"].isna()].index

    fallback = df_long.loc[faltantes_idx, ["Period", "Name_key"]].reset_index().merge(
        asistencia_total,
        on=["Period", "Name_key"],
        how="left",
    ).set_index("index")

    cols_to_fill = ["Afirmativo", "En_Contra", "Total_Votaciones", "Asistencia"]
    for col in cols_to_fill:
        merged.loc[faltantes_idx, col] = fallback[col]

    sin_match_final = merged["Asistencia"].isna().sum()
    print(f"Filas sin emparejar tras el fallback por Name_key: {sin_match_final} de {len(merged)}")

    if sin_match_final > 0:
        print()
        print("Ejemplos de diputados que no lograron emparejarse (revisar nombres manualmente):")
        display(merged.loc[merged["Asistencia"].isna(), ["Period", "Name"]].drop_duplicates())



## 6. Resultado final

`merged` contiene la base long original con las columnas de asistencia
agregadas: `Afirmativo`, `En_Contra`, `Total_Votaciones`, `Asistencia`.


In [ ]:

merged_final = merged.drop(columns=["Name_key"])
merged_final.head(20)


In [ ]:

# Guardar resultado
OUT_PATH = DATA_DIR.parent / "ideologia_con_asistencia.csv"
merged_final.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Guardado en: {OUT_PATH.resolve()}")
